In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import pandas as pd

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

list_jobs = []

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

driver.get("https://www.naukrigulf.com/search-jobs")
key_box = wait.until(EC.element_to_be_clickable((By.ID, "qsbKey")))
loc_box = wait.until(EC.element_to_be_clickable((By.ID, "qsbLoc")))
key_box.send_keys("Data Engineer")
loc_box.send_keys("Egypt")

search_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="ngQsbForm"]/input')))
search_btn.click()


jobs = driver.find_elements(By.CLASS_NAME, "srp-tuple")

print("Found:", len(jobs))
for card in jobs:
    print(card.text[:120])
        
        
for job in jobs:
    try:
      job_title = job.find_element(By.CLASS_NAME, 'designation-title').text
    except:
      job_title = 'No job title'
    try:
      Company = job.find_element(By.CLASS_NAME, 'info-org').text
    except:
      Company = 'No Company'
    try:
      Location = job.find_element(By.CLASS_NAME, 'info-loc').text
    except:
      Location = 'No Location'
    try:
      Experience = job.find_element(By.CLASS_NAME, 'info-exp').text
    except:
      Experience = 'No Experience'
    try:
       Description = job.find_element(By.CLASS_NAME, 'description').text
    except:
      Description = 'No Exp'  
    

    list_jobs.append({
        'job_title': job_title,
        'Company': Company,
        'Location': Location,
        'Experience': Experience,
        'Description': Description
    })
    Next_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="pagination"]/ul/li[5]/span')))
    Next_btn.click()
      
driver.quit()
 
df = pd.DataFrame(list_jobs)
df.to_csv('Jobs.csv', index=False)

Found: 0


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time # هنحتاجه عشان نعمل وقفة صغيرة بين الصفحات

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

list_jobs = []

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

try:
    driver.get("https://www.naukrigulf.com/search-jobs")
    
    # 1. إدخال بيانات البحث
    key_box = wait.until(EC.element_to_be_clickable((By.ID, "qsbKey")))
    loc_box = wait.until(EC.element_to_be_clickable((By.ID, "qsbLoc")))
    key_box.send_keys("Data Engineer")
    loc_box.send_keys("Egypt")

    search_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="ngQsbForm"]/input')))
    search_btn.click()

    # 2. اللوب الخاص بالصفحات (مثلاً هنخليه يقلب 3 صفحات كبداية)
    num_pages_to_scrape = 3 
    
    for page in range(num_pages_to_scrape):
        # مهم جداً: نستنى لحد ما كروت الوظايف تظهر في الصفحة الجديدة
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "srp-tuple")))
        time.sleep(2) # ثانية إضافية عشان نتأكد إن كل الداتا حملت (Javascript)
        
        jobs = driver.find_elements(By.CLASS_NAME, "srp-tuple")
        print(f"--- Page {page + 1}: Found {len(jobs)} jobs ---")
        
        # 3. اللوب الخاص باستخراج البيانات من كل صفحة
        for job in jobs:
            try: job_title = job.find_element(By.CLASS_NAME, 'designation-title').text
            except: job_title = 'No job title'
            
            try: Company = job.find_element(By.CLASS_NAME, 'info-org').text
            except: Company = 'No Company'
            
            try: Location = job.find_element(By.CLASS_NAME, 'info-loc').text
            except: Location = 'No Location'
            
            try: Experience = job.find_element(By.CLASS_NAME, 'info-exp').text
            except: Experience = 'No Experience'
            
            try: Description = job.find_element(By.CLASS_NAME, 'description').text
            except: Description = 'No Description'  

            list_jobs.append({
                'job_title': job_title,
                'Company': Company,
                'Location': Location,
                'Experience': Experience,
                'Description': Description
            })
        
        # 4. الضغط على زرار الصفحة التالية (Next)
        try:
            # استخدمت مسار أدق شوية لزرار الـ Next عشان ما يضربش لو ترتيب الـ li اتغير
            Next_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//a[contains(@class, "next")] | //*[@id="pagination"]/ul/li[last()]/span')))
            Next_btn.click()
        except Exception as e:
            print("No more pages available or Next button not found.")
            break # لو ملقاش الزرار يوقف اللوب

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    driver.quit()

# 5. حفظ البيانات
df = pd.DataFrame(list_jobs)
df.to_csv('Jobs.csv', index=False)
print(f"Successfully scraped {len(df)} jobs and saved to Jobs.csv!")

--- Page 1: Found 30 jobs ---
--- Page 2: Found 30 jobs ---
--- Page 3: Found 27 jobs ---
No more pages available or Next button not found.
Successfully scraped 87 jobs and saved to Jobs.csv!


In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException, StaleElementReferenceException
import pandas as pd

# ── Setup ────────────────────────────────────────────────────────────────────

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

list_jobs = []

# ── Helper: safely get text from a card ──────────────────────────────────────

def get_text(element, class_name, fallback="N/A"):
    try:
        return element.find_element(By.CLASS_NAME, class_name).text
    except (NoSuchElementException, StaleElementReferenceException):
        return fallback

# ── Search ───────────────────────────────────────────────────────────────────

try:
    driver.get("https://www.naukrigulf.com/search-jobs")

    wait.until(EC.element_to_be_clickable((By.ID, "qsbKey"))).send_keys("Data Engineer")
    wait.until(EC.element_to_be_clickable((By.ID, "qsbLoc"))).send_keys("Egypt")
    wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="ngQsbForm"]/input'))).click()

    # ── Paginate ──────────────────────────────────────────────────────────────

    while True:
        # Wait for cards to be present
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "srp-tuple")))

        # Re-fetch card count fresh (never cache the list)
        num_jobs = len(driver.find_elements(By.CLASS_NAME, "srp-tuple"))
        print(f"Found {num_jobs} jobs on this page")

        for i in range(num_jobs):
            try:
                # Re-query the specific card fresh on every iteration
                # This is the core fix: don't reuse a stored reference
                job = driver.find_elements(By.CLASS_NAME, "srp-tuple")[i]

                list_jobs.append({
                    "job_title":   get_text(job, "designation-title", "No Title"),
                    "company":     get_text(job, "info-org",          "No Company"),
                    "location":    get_text(job, "info-loc",          "No Location"),
                    "experience":  get_text(job, "info-exp",          "No Experience"),
                    "description": get_text(job, "description",       "No Description"),
                })

            except StaleElementReferenceException:
                print(f"Card {i} went stale, skipping.")
                continue

        # ── Next page ─────────────────────────────────────────────────────────

        try:
            next_btn = wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="pagination"]/ul/li[5]/span'))
            )
            next_btn.click()
        except TimeoutException:
            print("No more pages.")
            break

finally:
    driver.quit()

# ── Save ──────────────────────────────────────────────────────────────────────

df = pd.DataFrame(list_jobs)
df.to_csv("Jobs.csv", index=False)
print(f"Saved {len(df)} jobs to Jobs.csv")


TimeoutException: Message: 
